In [ ]:
import numpy as np
import pyvista as pv

from phd_helpers.AbaqusPreprocessing import bone_surface_patch_nodes, AbaqusInpBuilder

In [4]:
bone = 'tpm'
mesh = pv.read(f'../volumetricMesh/cgal-box/{bone}-final-postioned-quad.vtu')

In [ ]:
WARNING: THE MEMORY LIMIT IS INSUFFICENT TO RUN THE PURE THREAD-BASED 
             SOLVER. MEMORY REQUIRED 68776 MB, MEMORY LIMIT 25770 MB. REVERTING 
             TO THE HYBRID SOLVER FOR OUT-OF-CORE SOLUTION.

In [ ]:
filename = r"C:\Users\scmo\Downloads/test"
#filename = 'test'

# OUPUT FILE PATH
output_inp = filename + ".inp"

# ABAQUS CLI INPUTS
inp_file = output_inp # path to input file i.e inputs/input1.inp
job_name = filename   # saves files to this path i.e. inputs/input1
CPUs = 8 # 4(=8tokens) performance won't get much better after 8(=12tokens)
setting = 'interactive'

# MESH PATHS
trapezium_vtu = '../volumetricMesh/cgal-box/tpm-final-postioned.vtu' #'../volumetricMesh/cgal-box/tpm-final-postioned-quad.vtu'
metacarpal_vtu ='../volumetricMesh/cgal-box/mc1-final-postioned.vtu' #'../volumetricMesh/cgal-box/mc1-final-postioned-quad.vtu'

# ELEMENT ORDER
element_order = 'linear' # or 'quad'

# ELEMENT TYPES
bone_element_type = "C3D4"
cartilage_element_type = "C3D4H"

# BONE PROPERTIES
bone_material = {
    "E": 1629, # MPa
    "nu": 0.4
}
bone_density=None

# CARTILAGE PROPERTIES
cartilage_material = {
    "C10": 0.091,
    "D1": 0.0         
}
cartilage_density=None

cartilage_friction = 0.0

# REGION IDs
bone_region_id=1
cartilage_region_id=2
cartilage_outer_tri_region_id=-2

# DISPLACEMENT / FORCE LIMITS
mc1_disp_x  = -0.80 # end analysis at this displacement     - starting point is 0.01mm from contact
#Forces = [10.0, 20.0]   # refine step time to hit these forces - would need to set user defined DT REFINEMENT - not worth it right now
max_force = 50.0    # end analysis at this force

# STEP PARAMS
total_step_time = abs(mc1_disp_x) # set to total displacement so that increment params don't have to change with displacement
initial_increment = 0.01          # starting point is 0.01mm from contact
min_increment = 1e-10
max_increment = 0.025

step_type = "STATIC"
nlgeom = "YES" # non-linear geometry
unsymm = "YES" # store unsymmetric matrix
convert_sdi = "NO" #force a new iteration if severe discontinuities occur during an iteration, regardless of the magnitude of the penetration and force errors

equil_iters = 16 # default=16 - upper limit on the number of consecutive equilibrium iterations (without severe discontinuities) (4)
sdi_iters = 15 # deafult=12 - maximum number of severe discontinuity iterations allowed in an increment if CONVERT SDI=NO (7)
increment_attemps = 5 # default=5 - maximum number of attempts allowed for an increment (8)




In [ ]:
b = AbaqusInpBuilder()

# SET ELEMENT TYPE AND REGION IDS
b.set_element_types(bone_element_type, cartilage_element_type)
b.set_region_ids(bone_region_id, cartilage_region_id, cartilage_outer_tri_region_id)

# MATERIALS
b.create_material("BONE", "elastic", bone_material, density=bone_density)
b.create_material("CARTILAGE", "neo_hooke", cartilage_material, density=cartilage_density)

# PARTS
b.add_part_from_vtu("tpm", trapezium_vtu, instance_name="tpm_INST")
b.add_part_from_vtu("mc1", metacarpal_vtu, instance_name="mc1_INST")
# get meshes for node sets that follow
tpm_mesh = b.parts["tpm"]["mesh"]
mc1_mesh = b.parts["mc1"]["mesh"]

# ELSETS
b.create_elset_from_region("tpm", bone_region_id, "tpm_BONE")
b.create_elset_from_region("tpm", cartilage_region_id, "tpm_CARTILAGE")
b.create_elset_from_region("mc1", bone_region_id, "mc1_BONE")
b.create_elset_from_region("mc1", cartilage_region_id, "mc1_CARTILAGE")

# SOLID SECTIONS
b.create_solid_section("tpm", "tpm_BONE", "BONE")
b.create_solid_section("tpm", "tpm_CARTILAGE", "CARTILAGE")
b.create_solid_section("mc1", "mc1_BONE", "BONE")
b.create_solid_section("mc1", "mc1_CARTILAGE", "CARTILAGE")

# SURFACES FOR BONE CONSTRAINTS
tpm_patch_nodes = bone_surface_patch_nodes(tpm_mesh, [-100, 0], "x_lims", True)
mc1_patch_nodes = bone_surface_patch_nodes(mc1_mesh, 10, "euclidean", True)

b.add_surface_from_nodes("tpm", tpm_patch_nodes, "tpm_PATCH_NODES", "tpm_PATCH_SURF")
b.add_surface_from_nodes("mc1", mc1_patch_nodes, "mc1_PATCH_NODES", "mc1_PATCH_SURF")

# RPs FOR SURFACE PATCH COUPLING
tpm_centroid = tpm_mesh.points.mean(axis=0)
mc1_centroid = mc1_mesh.points.mean(axis=0)
b.create_reference_point("RP_tpm", node_id=9000001, xyz=tpm_centroid)
b.create_reference_point("RP_mc1", node_id=9000002, xyz=mc1_centroid)

# RP-SURFACE COUPLING
b.add_rp_surface_coupling("CP_tpm", "RP_tpm", "tpm", "tpm_PATCH_SURF", coupling_type="KINEMATIC")
b.add_rp_surface_coupling("CP_mc1", "RP_mc1", "mc1", "mc1_PATCH_SURF", coupling_type="KINEMATIC")

# CARTILAGE SURFACES FOR CONTACT
b.add_surface_from_region_id("tpm", cartilage_outer_tri_region_id, "tpm_CART_SURF")
b.add_surface_from_region_id("mc1", cartilage_outer_tri_region_id, "mc1_CART_SURF")

# CONTACT
b.set_contact(
    interaction_name = "CART_CONTACT",
    friction = cartilage_friction,
    surfaces = [("tpm", "tpm_CART_SURF", "mc1", "mc1_CART_SURF")], # [(part1, surface1, part2, surface2)]
)

# STEP
step_name = "MOVE"
b.create_step(
    step_name = step_name,
    step_type = step_type,
    initial_increment_size = initial_increment,     # starting point is 0.01mm from contact
    total_step_size = total_step_time, # set to total displacement so that increment params don't have to change with displacement
    min_increment_size = min_increment,
    max_increment_size = max_increment, 
    nlgeom = nlgeom,
    convert_sdi = convert_sdi,
    unsymm=unsymm
)

# CONTROLS
b.add_control_lines(
    step_name,
    [
    '*CONTROLS, PARAMETERS=TIME INCREMENTATION',
    f',,,{equil_iters},,,{sdi_iters},{increment_attemps},,,,,'
]
)

# BOUNDARY CONDITIONS
b.set_bc(
    step_name,
    node_set='RP_tpm',
    op='MOD', # MOD-change current BCs, NEW-replace all current BCs
    U1=0, U2=0, U3=0, UR1=0, UR2=0, UR3=0
)
b.set_bc(
    step_name,
    op='MOD',
    node_set='RP_mc1',
    U1=mc1_disp_x, U2=0, U3=0, UR1=0, UR2=0, UR3=0
)

# HISTORY OUTPUTS
b.add_history_output_lines(
    step_name,
    [
    "*OUTPUT, HISTORY, SENSOR, NAME=mc1_RF1",
    "*NODE OUTPUT, NSET=RP_mc1",
    "RF1"
]
)
b.add_history_output_lines(
    step_name,
    [
    "*OUTPUT, HISTORY",
    "*CONTACT OUTPUT, SURFACE=tpm_INST.tpm_CART_SURF",
    "CAREA"
]
)

# FIELD OUTPUTS
b.add_field_output_lines(
    step_name,
    [
    "*OUTPUT, FIELD",
    "**",
    "*NODE OUTPUT",
    "U, COORD",
    "**",
    "*ELEMENT OUTPUT, POSITION=INTEGRATION POINTS",
    "S, LE, COORD",
    "**",
    "*CONTACT OUTPUT",
    "CSTRESS, CDISP, CSTATUS, CNAREA"
])

# STEP CONTROLS
#for F in Forces:
#    b.add_step_control_lines(
#        step_name,
#        [
#        f"*STEP CONTROL, NAME=REFINE_ON_RF{F:.0f}, ACTION=CONTINUE, DT REFINEMENT=YES", # need to set user defined DT REFINEMENT - not worth it right now
#        f"mc1_RF1, AbsMax, {float(F)}",
#    ])


b.add_step_control_lines(
    step_name,
    [
    f"*STEP CONTROL, NAME=STOP_ON_RF{max_force:.0f}, ACTION=END ANALYSIS, DT REFINEMENT=AUTO",
    f"mc1_RF1, AbsMax, {float(max_force)}",
])

# WRITE INPUT FILE
b.write_input_file(output_inp)

'C:\\Users\\scmo\\Downloads/test.inp'

# img of of FEA setup for presentation

In [57]:
tpm_mesh['patch_node'] = np.zeros(tpm_mesh.n_points)
tpm_mesh['patch_node'][tpm_patch_nodes] = 1.
mc1_mesh['patch_node'] = np.zeros(mc1_mesh.n_points)
mc1_mesh['patch_node'][mc1_patch_nodes] = 1.

tpm_shell = tpm_mesh.extract_cells_by_type(5)
mc1_shell = mc1_mesh.extract_cells_by_type(5)
tpm_shell['shell_id'] = np.arange(tpm_shell.n_cells)
mc1_shell['shell_id'] = np.arange(mc1_shell.n_cells)

tpm_patch = tpm_shell.extract_points(np.where(tpm_shell['patch_node']==1)[0], adjacent_cells=False)
mc1_patch = mc1_shell.extract_points(np.where(mc1_shell['patch_node']==1)[0], adjacent_cells=False)

tpm_notpatch = tpm_shell.extract_cells(tpm_patch['shell_id'], invert=True)
mc1_notpatch = mc1_shell.extract_cells(mc1_patch['shell_id'], invert=True)

In [ ]:
pl = pv.Plotter()
pl.add_mesh(tpm_patch, color='gray')
pl.add_mesh(mc1_patch, color='gray')

pl.add_mesh(tpm_notpatch, scalars=-tpm_notpatch['region_id'], cmap=['white', 'coral'], show_scalar_bar=False)
pl.add_mesh(mc1_notpatch, scalars=-mc1_notpatch['region_id'], cmap=['white', 'coral'], show_scalar_bar=False)

pl.camera_position = [(14.515150056815644, 3.9728833399677272, 42.3288649427451),
 (15.981080904848982, 0.7901038546155648, -0.7259656771213434),
 (-0.032772519412218225, 0.9966604209485677, -0.07479282910814256)]
pl.show()

Widget(value='<iframe src="http://localhost:64798/index.html?ui=P_0x216a6ddc7d0_15&reconnect=auto" class="pyvi…

pl.screenshot(
    r"C:\Users\scmo\OneDrive - University of Leeds\PhD-wrist\Presentations\meeting-2026-02-24/initalPosition",
    return_img=False,
    transparent_background=True,
    scale=4
)